> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 9 · Notebook 03 — The pairs strategy, end to end

**Sessions:** S3 (The pairs strategy, end to end) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. Build a z-score whose look-back is tied to the half-life.
2. Write the pairs state machine: entry, exit, stop, time stop and no re-entry after a stop.
3. Compute the P&L of both legs, with trading costs and borrow fees.
4. Trade out of sample, then watch what a dying relationship does to a rolling and a fixed z-score.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. Formation, then trading

Estimate everything on a **formation** period (here the first 750 days) and trade only after it. The spread uses the formation hedge ratio; the z-score look-back is about three half-lives, long enough to estimate a mean and a volatility, short enough to follow slow drift.

In [ ]:
pair = p.cointegrated_pair()                       # true β 1.5, spread half-life 7 days
F = 750
eg = p.engle_granger(pair.y.iloc[:F], pair.x.iloc[:F])
hl = eg["half_life"]
W = round(3 * hl)
spread = pair.y - eg["beta"] * pair.x - eg["alpha"]
print(f"formation: β {eg['beta']:.3f}, p-value {eg['pvalue']:.1e}, half-life {hl:.1f} days → z look-back {W} days")

## 2. The z-score

`z = (s − rolling mean) / rolling std` over the last `window` values **including today** (std with ddof=1, the pandas default), NaN during the warm-up. `pd.Series.rolling(window)` gives you both the mean and the std.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def rolling_zscore(spread, window):
    s = pd.Series(np.asarray(spread, dtype=float))
    r = s.rolling(window)
    return ((s - r.mean()) / r.std()).to_numpy()

cases = [(spread, W), ([1.0, 2.0, 4.0, 3.0, 5.0], 3)]
mine = [p.attempt(rolling_zscore, *c) for c in cases]
mine = p.check("rolling_zscore", mine, [p.rolling_zscore(*c) for c in cases])
z = mine[0]
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(spread.index[F:], z[F:], lw=1)
for lvl, c in ((2, "#eb6834"), (0.5, "#1baf7a"), (4, "#e34948")):
    ax.axhline(lvl, color=c, lw=0.8, ls="--"); ax.axhline(-lvl, color=c, lw=0.8, ls="--")
ax.set_title("trading period: z-score with entry ±2, exit ±0.5, stop ±4"); plt.show()

## 3. The state machine

`+1` is long the spread (long `y`, short `β·x`), `−1` short it. Decisions are made at the close and filled on the next bar.

* **Flat:** enter `−sign(z)` when `entry < |z| < stop` (beyond the stop, the relationship may already be breaking).
* **In a trade** (count `held` bars after the entry bar): exit when `|z| < exit_`. **Stop out** when `|z| > stop` or, with a time stop, when `held >= max_hold`.
* After a stop or time stop, **don't re-enter** until `|z|` has come back below `entry`: a pair that just stopped you out is not a fresh signal.
* A NaN z-score (warm-up) leaves the position unchanged.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pairs_positions(z, entry=2.0, exit_=0.5, stop=4.0, max_hold=None):
    z = np.asarray(z, dtype=float)
    pos, held, blocked = np.zeros(z.size), 0, False
    for t in range(z.size):
        cur = pos[t - 1] if t else 0.0
        if np.isnan(z[t]):
            pos[t] = cur                                  # no z-score yet: stay as you are
            continue
        if cur == 0:
            blocked = blocked and abs(z[t]) >= entry      # after a stop, wait until |z| is back below entry
            if not blocked and entry < abs(z[t]) < stop:
                cur, held = -np.sign(z[t]), 0
        else:
            held += 1
            if abs(z[t]) < exit_:
                cur = 0.0
            elif abs(z[t]) > stop or (max_hold is not None and held >= max_hold):
                cur, blocked = 0.0, True
        pos[t] = cur
    return pos

toy = [np.nan, 0.3, 2.5, 1.8, 0.4, -2.2, -4.5, -3.0, -1.5, -2.5, 0.1]
slow = [2.5, 2.4, 2.3, 2.2, 2.1, 2.6, 1.0, 2.5]
cases = [(toy,), (slow, 2.0, 0.5, 4.0, 3), (z, 2.0, 0.5, 4.0, W)]
mine = [p.attempt(pairs_positions, *c) for c in cases]
mine = p.check("pairs_positions", mine, [p.pairs_positions(*c) for c in cases])
display(pd.DataFrame({"z": toy, "position": mine[0]}).T)
display(pd.DataFrame({"z": slow, "position (time stop 3 bars)": mine[1]}).T)

In the first table, the stop at `z = −4.5` blocks the next bar (`−3.0` is still beyond 2); `−1.5` clears the block and `−2.5` is a fresh entry. In the second, the time stop closes the trade after 3 bars and nothing re-enters until `|z|` has dipped below 2.

## 4. P&L with both legs, costs and borrow

Work in **log prices**, so leg returns are `Δy` and `Δx`. A unit spread position holds 1 of `y` against `β` of `x`: gross capital `1 + |β|`. The position decided yesterday earns

`ret = held · (Δy − β·Δx) / (1 + |β|)`

(use `np.nan_to_num`: the first `Δ` is NaN). Costs: `|Δpos| · cost_bps/1e4 · 2` legs, charged on the bar the trade fills. Borrow: while you hold a position you pay `borrow_bps_year/1e4/252` a day on the **short** leg, whose weight is `1/(1 + |β|)` when short the spread (short `y`) and `|β|/(1 + |β|)` when long it (short `x`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pair_pnl(pos, y, x, beta, cost_bps=5.0, borrow_bps_year=50.0):
    pos = np.asarray(pos, dtype=float)
    dy, dx = np.diff(np.asarray(y, float), prepend=np.nan), np.diff(np.asarray(x, float), prepend=np.nan)
    held = np.concatenate([[0.0], pos[:-1]])              # decided at yesterday's close, held today
    beta = np.asarray(beta, dtype=float)
    gross = 1 + np.abs(beta)
    ret = np.nan_to_num(held * (dy - beta * dx) / gross)
    short_w = np.where(held < 0, 1 / gross, np.where(held > 0, np.abs(beta) / gross, 0.0))
    trade_cost = np.abs(np.diff(pos, prepend=0.0)) * cost_bps / 1e4 * 2
    trade_cost = np.concatenate([[0.0], trade_cost[:-1]])  # the trade fills on the next bar
    return ret - trade_cost - short_w * borrow_bps_year / 1e4 / 252

pos = mine[2]
cases = [(pos, pair.y, pair.x, eg["beta"]), (pos, pair.y, pair.x, eg["beta"], 0.0, 300.0),
         ([0, 1, 1, -1, 0], [0, 0.01, 0.03, 0.02, 0.02], [0, 0.01, 0.01, 0.0, 0.01], 1.0, 10.0, 252.0)]
mine = [p.attempt(pair_pnl, *c) for c in cases]
mine = p.check("pair_pnl", mine, [p.pair_pnl(*c) for c in cases])
pnl = mine[0][F:]
tr = pos[F - 1:]
print(f"out of sample ({len(pnl)} days): Sharpe {p.sharpe(pnl):.2f}, total {pnl.sum():.1%}, "
      f"{int(((tr[1:] != 0) & (tr[1:] != tr[:-1])).sum())} trades, in a position {np.mean(pos[F:] != 0):.0%} of the days")
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(pair.index[F:], np.cumsum(pnl)); ax.set_title("cumulative P&L of the pair, out of sample (5 bp per leg, 50 bp/yr borrow)"); plt.show()

## 5. Costs decide

Every round trip pays four fills (two legs, in and out). A fast-reverting pair trades often, so the cost per fill matters far more than the borrow fee:

In [ ]:
grid = {c: {b: p.sharpe(p.pair_pnl(pos, pair.y, pair.x, eg["beta"], c, b)[F:]) for b in (0, 50, 300)} for c in (0, 5, 10, 20)}
pd.DataFrame(grid).T.rename_axis("cost per leg (bp) ↓ · borrow (bp/yr) →").round(2)

Sharpe 1.9 before costs, 1.2 at 5 bp a leg, 0.5 at 10 bp and negative at 20 bp. Even a hard-to-borrow fee of 300 bp a year costs only about 0.1 of Sharpe here.

## 6. When the relationship dies

`p.breaking_pair()` is cointegrated for 900 days, then its spread becomes a random walk (a merger, a new business line). Trade it from day 750 with two yardsticks: the **rolling** z-score, and a **formation** z-score that keeps the formation mean and std fixed. Try both with and without a time stop of three half-lives.

In [ ]:
bp = p.breaking_pair()
eb = p.engle_granger(bp.y.iloc[:F], bp.x.iloc[:F])
Wb = round(3 * eb["half_life"])
sb = bp.y - eb["beta"] * bp.x - eb["alpha"]
yardsticks = {"rolling z": p.rolling_zscore(sb, Wb),
              "formation z": ((sb - sb.iloc[:F].mean()) / sb.iloc[:F].std()).to_numpy()}
rows = {}
for name, zz in yardsticks.items():
    for mh in (None, Wb):
        pl = p.pair_pnl(p.pairs_positions(zz, max_hold=mh), bp.y, bp.x, eb["beta"])
        rows[(name, "no time stop" if mh is None else f"time stop {mh} bars")] = {
            "P&L 750–900 (healthy)": pl[F:900].sum(), "P&L after break": pl[900:].sum(),
            "max |z| after break": np.nanmax(np.abs(zz[900:]))}
display(pd.DataFrame(rows).T.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True)
for ax, (name, zz) in zip(axes, yardsticks.items()):
    ax.plot(bp.index[F:], zz[F:], lw=0.8); ax.axvline(bp.index[900], color="black", lw=1)
    ax.axhline(4, color="#e34948", ls="--", lw=0.8); ax.axhline(-4, color="#e34948", ls="--", lw=0.8); ax.set_title(name)
    ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

The rolling z-score **re-scales** to the wider spread: after the break it never reaches 4, so the stop never fires and the strategy keeps trading a dead pair. With a time stop it does worse, because every time stop is followed by a fresh entry. The formation z-score sees the spread run away (|z| up to 16): the stop fires and blocks re-entry, and with a time stop the post-break loss disappears. Stops need a fixed yardstick, and even then they are a last line of defence: notebook 07 **monitors** the relationship and retires the pair.

## Wrap-up

* Formation first, then trade; never estimate on the data you trade.
* Look-back ≈ 3 half-lives; time stop ≈ 3 half-lives; no re-entry after a stop.
* Two legs, in and out: costs dominate borrow for a fast pair.
* A rolling z-score hides a break.
* Graded version: `labs/part09/week31_pairs` (`rolling_zscore`, `pairs_positions`, `pair_pnl`) and Clinic W1.